In [1]:
import re
import importlib
import requests
import datetime
import json
from pathlib import Path

from tqdm import tqdm
import pandas as pd
import numpy as np

import lib


importlib.reload(lib)

log = lib.getLogger(Path().cwd().name)
working_directory = Path().cwd()
export_clean_directory = working_directory / 'clean'

In [2]:
categories_path = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_categories_CLEAN.csv'
export_categories_path = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_categories.csv'

In [3]:
df = lib.read_data(categories_path)

PortfolioLogger.lib.tools: INFO: Encoding: utf-8
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from my_books_categories_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10d436980> took 0.055 secs to complete.


In [4]:
df

,Unnamed: 0,Book Id,Category
0,0,18400112,Language Arts & Disciplines
1,1,62047984,Language Arts & Disciplines
2,2,58416952,Fiction
3,3,58416952,High Fantasy
4,4,58416952,Series:Hierarchy
...,...,...,...
3765,3765,43848929,Social Science
3766,3766,43848929,Strangers
3767,3767,43848929,Threat (Psychology)
3768,3768,43848929,Trust


In [9]:
categories = df['Category'].unique().tolist()

In [10]:
len(categories)

2032

In [11]:
categories

['Language Arts & Disciplines',
 'Fiction',
 'High Fantasy',
 'Series:Hierarchy',
 'Fantasy',
 'Fantasy Fiction',
 'General',
 'Action & Adventure',
 'Genius',
 'Science Fiction',
 'Gifted Persons',
 'Hallucinations And Illusions',
 'Multiple Personality',
 'Short Stories',
 'Abuelos (Hombres)',
 'Aptitudes Motoras',
 'Bibliotecarios',
 'Bibliothécaires',
 "Children'S Fiction",
 'Clumsiness',
 'Cuentos Humorosos',
 'Etc. Pour La Jeunesse',
 'Grandfathers',
 'Grandparents',
 'Grands-Pères',
 'Humorous Stories',
 'Juvenile Fiction',
 'Juvenile Fiction / Action & Adventure / General',
 'Juvenile Fiction / Fantasy & Magic',
 'Librarians',
 'Maladresse',
 'Materiales En Español',
 'New York Times Bestseller',
 'Nouvelles',
 'Novela Fantástica',
 'Novela Juvenil',
 'Romans',
 'English Literature',
 'Fiction / Fantasy / Action & Adventure',
 'Fiction / Fantasy / Epic',
 'Grail',
 'Quests (Expeditions)',
 'Vampires',
 'Dark Fantasy',
 'Epic',
 'Geographical Myths',
 'Historical',
 'Imaginary P

In [25]:
counts = df['Category'].value_counts()
counts

Category
Fiction                        129
General                         80
New York Times Bestseller       77
Language Arts & Disciplines     74
Fantasy                         49
                              ... 
Selenium                         1
Robots In Fiction                1
Robotics                         1
Robopsychology                   1
Trust                            1
Name: count, Length: 2032, dtype: int64

In [21]:
len(counts)

Index(['Fiction', 'General', 'New York Times Bestseller',
       'Language Arts & Disciplines', 'Fantasy', 'Science Fiction',
       'Nouvelles', 'Romans', 'Large Type Books', 'Fantasy Fiction',
       ...
       'Supercomputers', 'Space-Based Solar Power', 'Space Stations',
       'Smear Campaigns', 'Shahada', 'Selenium', 'Robots In Fiction',
       'Robotics', 'Robopsychology', 'Trust'],
      dtype='object', name='Category', length=2032)

In [22]:
i = 0
top = {}
for i, (col, item) in enumerate(counts.items()):
    top[col] = item
    if i == 100:
        break

In [23]:
top

{'Fiction': 129,
 'General': 80,
 'New York Times Bestseller': 77,
 'Language Arts & Disciplines': 74,
 'Fantasy': 49,
 'Science Fiction': 38,
 'Nouvelles': 34,
 'Romans': 32,
 'Large Type Books': 26,
 'Fantasy Fiction': 25,
 'New York Times Reviewed': 24,
 'Action & Adventure': 20,
 'Magic': 20,
 'English Literature': 19,
 'Epic': 18,
 'American Literature': 18,
 "Children'S Fiction": 17,
 'Reading Level-Grade 11': 17,
 'Reading Level-Grade 12': 17,
 'Open Library Staff Picks': 16,
 'Long Now Manual For Civilization': 16,
 'Ficción': 15,
 'Novela': 15,
 'Psychological': 14,
 'Thrillers': 14,
 'Historical': 13,
 'Literary': 13,
 'Friendship': 13,
 'Reading Level-Grade 10': 13,
 'Juvenile Fiction': 12,
 'Imaginary Places': 12,
 'Adventure': 11,
 'Reading Level-Grade 9': 11,
 'Good And Evil': 11,
 'Literature': 11,
 'Psychology': 11,
 'Survival': 11,
 'British And Irish Fiction (Fictional Works By One Author)': 10,
 'Roman': 10,
 'American Science Fiction': 9,
 'Suspense': 9,
 'Interpers

In [46]:
to_remove = [
    'Fiction',
    'General',
    'New York Times Bestseller',
    'Language Arts & Disciplines',
    'Romans',
    'Large Type Books',
    'New York Times Reviewed',
    'Reading Level-Grade 11',
    'Reading Level-Grade 12',
    'Open Library Staff Picks',
    'Long Now Manual For Civilization',
    'Ficción',
    'Novela',
    'Reading Level-Grade 10',
    'Reading Level-Grade 9'
] + counts[counts == 1].index.tolist()
to_remove

['Fiction',
 'General',
 'New York Times Bestseller',
 'Language Arts & Disciplines',
 'Romans',
 'Large Type Books',
 'New York Times Reviewed',
 'Reading Level-Grade 11',
 'Reading Level-Grade 12',
 'Open Library Staff Picks',
 'Long Now Manual For Civilization',
 'Ficción',
 'Novela',
 'Reading Level-Grade 10',
 'Reading Level-Grade 9',
 'Regression (Psychology)',
 'Whales In Literature',
 'Academic Literacy',
 'Father Time (Symbolic Character)',
 'Whales',
 'Achab (Personnage Fictif)',
 'Shipwreck Survival',
 '813/.3',
 'Regression',
 'Freedom',
 'Whalers (Persons)',
 'Walfang',
 'Symbolism',
 'Adventure Fiction',
 'Television Personalities',
 'Political Corruption',
 'Zhang Pian Xiao Shuo',
 'Whales--Fiction',
 'Whaling',
 'William',
 'Whaling In Literature',
 'Shipwrecks--Fiction',
 'Whaling Ships',
 'Whaling Ships--Fiction',
 'Whaling--Fiction',
 'Performing Arts / Comedy',
 'Sweden',
 'Syllabi',
 'Relationships',
 'Shipwreck',
 'Ahab',
 'Captain (Fictitious Character)--Fiction'

In [47]:
drop_mask = df[df['Category'].isin(to_remove)]

In [57]:
df.drop(drop_mask.index, axis=0)

,Unnamed: 0,Book Id,Category
3,3,58416952,High Fantasy
8,8,22878967,Fantasy
9,9,22878967,Fantasy Fiction
13,13,37640636,Action & Adventure
14,14,37640636,Fantasy
...,...,...,...
3755,3755,39863488,Thrillers
3757,3757,43848929,Conduct Of Life
3759,3759,43848929,Interpersonal Relations
3764,3764,43848929,Social Psychology


In [58]:
df = df.drop(drop_mask.index, axis=0)

In [59]:
lib.export_data(export_categories_path, df)

PortfolioLogger.lib.tools: INFO: Exporting 4782 elements (0.13 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_categories_CLEAN_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10d436ac0> took 0.006 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_categories_CLEAN_CLEAN.csv')